# LeetCode 3304 — Find the K-th Character in String Game I bằng Python

Link đề: https://leetcode.com/problems/find-the-k-th-character-in-string-game-i/ · Độ khó: **Easy** · Chủ đề: Simulation, Recursion, Bit Manipulation, Math

Skeleton LeetCode:

```python
class Solution:
    def kthCharacter(self, k: int) -> str:
        ...
```

Ghi chú: đề và ví dụ được dịch từ đề gốc (Weekly Contest 417); mọi lời giải trong notebook đã được chạy và kiểm chứng bằng code thật.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: chuỗi `word` bắt đầu là `"a"`, mỗi lần thao tác ta **đổi mỗi ký tự thành ký tự kế tiếp trong bảng chữ cái** rồi **nối** bản đổi đó vào cuối `word` (nên độ dài **gấp đôi** sau mỗi bước). Ta cần ký tự thứ `k`.
2. Nắm các công cụ Python cần thiết: `ord` / `chr` (đổi ký tự ↔ số), toán tử `%` để "quay vòng" `z → a`, list comprehension, `list.extend`, và các phép bit (`>>`, `bit_length`, `bit_count`).
3. Đi từ cách **mô phỏng bằng chuỗi** → **mô phỏng bằng độ lệch số (0..25)** → **chia đôi chỉ số (đệ quy / vòng lặp)** → **công thức đếm bit `1`** của `k - 1`.
4. Hiểu mấu chốt: nửa sau của `word` chính là nửa đầu **cộng thêm 1**, nên ký tự thứ `k` chỉ phụ thuộc "số lần ta rơi vào nửa sau" khi chia đôi liên tục — con số đó là số bit `1` của `k - 1`.
5. Tránh các lỗi kinh điển: lệch chỉ số `k` và `k - 1`, quên quay vòng `z → a`, vừa lặp vừa nối vào chính list đang lặp, và quên trừ `1` trong công thức bit.

## 2. Đề bài & ví dụ

Alice và Bob chơi một trò chơi. Ban đầu Alice có chuỗi `word = "a"`. Cho một số nguyên **dương** `k`.

Bob yêu cầu Alice thực hiện thao tác sau **mãi mãi**:

- Tạo một chuỗi mới bằng cách **đổi mỗi ký tự** của `word` thành ký tự **kế tiếp** trong bảng chữ cái tiếng Anh (`z` đổi thành `a`), rồi **nối** chuỗi mới đó vào **sau `word` gốc**.

Ví dụ: thao tác trên `"c"` cho `"cd"`; thao tác trên `"zb"` cho `"zbac"`.

Trả về ký tự thứ `k` (đếm từ 1) của `word` sau khi đã làm đủ số thao tác để `word` có **ít nhất** `k` ký tự.

**Ví dụ 1**

- Input: `k = 5`
- Output: `"b"`
- Giải thích: `"a"` → `"ab"` → `"abbc"` → `"abbcbccd"`. Ký tự thứ 5 là `b`.

**Ví dụ 2**

- Input: `k = 10`
- Output: `"c"`

**Ràng buộc:** `1 <= k <= 500`.

In [1]:
examples = [
    (5, "b"),
    (10, "c"),
]
for k, expected in examples:
    print("k =", k, "-> kỳ vọng", repr(expected))

k = 5 -> kỳ vọng 'b'
k = 10 -> kỳ vọng 'c'


## 3. Kiến thức nền

### 3.1. `ord` và `chr`: đổi ký tự ↔ số

Máy tính lưu mỗi ký tự dưới dạng một con số (mã). `ord("a")` cho số của ký tự `a` (là `97`), và `chr(97)` làm ngược lại. Các chữ cái thường nằm **liền nhau**: `a=97, b=98, ..., z=122`. Nên "chữ kế tiếp" chính là `mã + 1`, và "chữ thứ `t` trong bảng" là `chr(ord("a") + t)` (với `t = 0` là `a`).

In [2]:
print(ord("a"), ord("b"), ord("z"))      # 97 98 122
print(chr(97), chr(98))                  # a b
t = 3
print(chr(ord("a") + t))                 # d  (chữ thứ 3 tính từ a, với a là chữ thứ 0)
print(ord("c") - ord("a"))               # 2  (c cách a hai bước = "độ lệch" của c)

97 98 122
a b
d
2


### 3.2. Toán tử `%` và việc "quay vòng" `z → a`

Bảng chữ cái có **26** chữ. Nếu ta lưu mỗi chữ dưới dạng **độ lệch** `0..25` (`a=0, ..., z=25`) thì "chữ kế tiếp" là `(độ_lệch + 1) % 26`. Phép `%` lấy **phần dư** khi chia cho 26: `25 + 1 = 26` có dư `0`, tức `z → a`. Cách này tránh phải xử lý riêng trường hợp `z`.

In [3]:
for x in [24, 25, 26, 27]:
    print(x, "% 26 =", x % 26)           # 24 25 0 1

z = ord("z") - ord("a")                  # độ lệch của z là 25
print(chr(ord("a") + (z + 1) % 26))      # a  (z quay về a)

24 % 26 = 24
25 % 26 = 25
26 % 26 = 0
27 % 26 = 1
a


### 3.3. List comprehension và `list.extend`

- `[f(x) for x in arr]` tạo **list mới** bằng cách áp `f` lên từng phần tử.
- `arr.extend(other)` nối **toàn bộ** `other` vào cuối `arr` (khác `append` chỉ thêm một phần tử).

Điều quan trọng: ở `word.extend([... for x in word])`, vế phải được **tính xong hoàn toàn** (một list mới) **rồi** mới nối vào `word`, nên không có chuyện "vừa lặp vừa sửa".

In [4]:
word = [0, 1]                            # độ lệch của "ab"
shifted = [(x + 1) % 26 for x in word]
print(shifted)                           # [1, 2]  (tức "bc")
word.extend(shifted)
print(word)                              # [0, 1, 1, 2]  (tức "abbc")

[1, 2]
[0, 1, 1, 2]


### 3.4. Độ dài gấp đôi và vòng `while len(word) < k`

Sau `m` thao tác độ dài là `2^m`: `1, 2, 4, 8, 16, ...`. Vì `k <= 500` nên chỉ cần tối đa `9` thao tác (`2^9 = 512 >= 500`). Ta lặp `while len(word) < k` để dừng ngay khi đủ dài.

In [5]:
length = 1
ops = 0
while length < 500:
    length *= 2
    ops += 1
print("cần", ops, "thao tác, độ dài", length)   # cần 9 thao tác, độ dài 512

cần 9 thao tác, độ dài 512


### 3.5. Các phép bit: `>>`, `bit_length`, `bit_count`

Ta sẽ cần ba thứ về số nhị phân (binary):

- `bin(n)` cho dạng nhị phân, ví dụ `bin(9) = '0b1001'`.
- `n.bit_length()` là số bit cần để biểu diễn `n` (`9 -> 4`).
- `n.bit_count()` là **số bit `1`** trong `n` (`9 -> 2`; còn gọi là *popcount*). Với Python cũ hơn 3.10 có thể dùng `bin(n).count("1")`.

In [6]:
n = 9
print(bin(n))                            # 0b1001
print(n.bit_length(), n.bit_count())     # 4 2
print(bin(n).count("1"))                 # 2  (cách tương đương)
print((1).bit_length(), (0).bit_count()) # 1 0

0b1001
4 2
2
1 0


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Viết ra vài thao tác đầu:

| Sau thao tác | `word` | độ dài |
|---|---|---|
| 0 | `a` | 1 |
| 1 | `ab` | 2 |
| 2 | `abbc` | 4 |
| 3 | `abbcbccd` | 8 |
| 4 | `abbcbccdbccdcdde` | 16 |

Nhìn bảng: mỗi dòng = dòng trước + (dòng trước với **mỗi chữ tăng 1**). Ví dụ dòng 4: nửa đầu `abbcbccd`, nửa sau `bccdcdde` — đúng là nửa đầu cộng 1 cho mỗi chữ.

Với `k = 5`: ký tự thứ 5 của `abbcbccd` là `b`. Với `k = 10`: ký tự thứ 10 của `abbcbccdbccdcdde` là `c`.

In [7]:
word = "a"
for step in range(1, 5):
    nxt = "".join(chr((ord(ch) - 97 + 1) % 26 + 97) for ch in word)   # mỗi chữ + 1
    word = word + nxt
    print(step, word, len(word))
print("k=5  ->", word[5 - 1])            # b
print("k=10 ->", word[10 - 1])           # c

1 ab 2
2 abbc 4
3 abbcbccd 8
4 abbcbccdbccdcdde 16
k=5  -> b
k=10 -> c


## 5. Cách 1 — Brute Force / baseline: mô phỏng bằng chuỗi

**Ý tưởng.** Làm đúng như đề: giữ `word` là một chuỗi, mỗi vòng tạo chuỗi "đổi mỗi chữ thành chữ kế tiếp" rồi nối vào. Dừng khi `len(word) >= k`, trả về `word[k - 1]` (Python đánh chỉ số từ 0 nên ký tự thứ `k` nằm ở chỉ số `k - 1`).

In [8]:
def shift_string(w: str) -> str:
    # đổi mỗi chữ thành chữ kế tiếp, z quay về a
    return "".join(chr((ord(ch) - ord("a") + 1) % 26 + ord("a")) for ch in w)


def kth_simulate_string(k: int) -> str:
    word = "a"
    while len(word) < k:
        word = word + shift_string(word)
    return word[k - 1]


for k, expected in examples:
    print(kth_simulate_string(k), "(kỳ vọng", expected, ")")
print(shift_string("zb"))                # ac  (kiểm tra ví dụ trong đề: zb -> zbac)

b (kỳ vọng b )
c (kỳ vọng c )
ac


**Độ phức tạp (Cách 1).** Độ dài cuối cùng nhỏ hơn `2k` (vì ta dừng ngay khi đủ `k`, và lần nhân đôi cuối từ `< k` thành `< 2k`). Tổng công việc là `1 + 2 + 4 + ... < 2k · 2` ⇒ **Time `O(k)`**, **Space `O(k)`**.

**Nút thắt:** ta dựng **cả** chuỗi dài hơn `k` chỉ để đọc **một** ký tự. Ta chỉ cần biết "ký tự thứ `k` được sinh ra từ ký tự nào, và đã bị cộng 1 bao nhiêu lần".

## 6. Ý tưởng thuật toán / bước đột phá

Gọi `W_m` là `word` sau `m` thao tác (độ dài `2^m`). Ta có `W_{m+1} = W_m + shift(W_m)`. Nghĩa là:

- Nếu chỉ số (đếm từ **0**) `i` nằm ở **nửa đầu** (`i < 2^m`): ký tự đó giống hệt ký tự ở `W_m`.
- Nếu `i` nằm ở **nửa sau** (`i >= 2^m`): ký tự đó = ký tự ở vị trí `i - 2^m` của `W_m` **cộng thêm 1**.

Vậy ta chỉ cần lặp: tìm `m` sao cho `i` ở nửa sau, **trừ** `2^m`, **cộng 1** vào "độ lệch", rồi làm tiếp với `i` nhỏ hơn đến khi `i = 0` (ký tự `a`). Mỗi lần "rơi vào nửa sau" ta cộng đúng 1 vào độ lệch. Phép trừ `2^m` chính là **xoá bit cao nhất** của `i`, nên số lần cộng 1 = số bit `1` của `i = k - 1`.

In [9]:
# kiểm tra "nửa sau = nửa đầu + 1" trên word sau 4 thao tác
word = "abbcbccdbccdcdde"
first, second = word[:8], word[8:]
print(first, second)
print(shift_string(first) == second)     # True
# i = 9 (k = 10) nằm ở nửa sau, vị trí tương ứng trong nửa đầu là 9 - 8 = 1
print(word[9], "=", shift_string(word[1]))   # c = c   (word[1] = 'b' cộng 1 thành 'c')

abbcbccd bccdcdde
True
c = c


## 7. Các cách tiếp theo

### Cách 2 — Mô phỏng bằng mảng độ lệch `0..25` (không dựng chuỗi)

**Giải thích từng đoạn.**

1. `word = [0]`: chữ `a` có độ lệch 0.
2. `while len(word) < k`: còn thiếu ký tự thì tiếp tục nhân đôi.
3. `word.extend([(x + 1) % 26 for x in word])`: nối bản "cộng 1, quay vòng mod 26".
4. `chr(ord("a") + word[k - 1])`: đổi độ lệch cuối cùng thành chữ.

Cách này gọn hơn vì làm việc với số, không phải ghép/cắt chuỗi.

In [10]:
def kth_offsets(k: int) -> str:
    word = [0]
    while len(word) < k:
        word.extend([(x + 1) % 26 for x in word])
    return chr(ord("a") + word[k - 1])


for k, expected in examples:
    print(kth_offsets(k), "(kỳ vọng", expected, ")")

b (kỳ vọng b )
c (kỳ vọng c )


In [11]:
# Code nộp LeetCode cho Cách 2 (đổi tên lớp thành Solution khi nộp)
class SolutionOffsets:
    def kthCharacter(self, k: int) -> str:
        word = [0]
        while len(word) < k:
            word.extend([(x + 1) % 26 for x in word])
        return chr(ord("a") + word[k - 1])


print(SolutionOffsets().kthCharacter(5), SolutionOffsets().kthCharacter(10))   # b c

b c


**Độ phức tạp Cách 2:** **Time `O(k)`**, **Space `O(k)`** (list độ lệch dài tới `< 2k`).

### Cách 3 — Chia đôi chỉ số (không dựng `word`)

**Giải thích từng đoạn.** Theo mục 6, ta không cần dựng `word` mà chỉ cần "lần ngược" nguồn gốc của ký tự thứ `k`:

1. `length`: nhân đôi từ `1` đến khi `length >= idx` — đó là độ dài của `word` đủ chứa vị trí `idx` (đếm từ 1).
2. Mỗi vòng: `half = length // 2`. Nếu `idx > half` thì `idx` ở **nửa sau**: `idx -= half` và `shift += 1`. Sau đó `length = half` (chỉ còn xét nửa tương ứng).
3. Khi `idx == 1` ta về tới chữ `a` ban đầu; kết quả là `a` cộng `shift`.

Dưới đây là hai cách viết: vòng lặp (tốt nhất cho người mới) và đệ quy.

In [12]:
def kth_halving(k: int) -> str:
    idx = k                              # vị trí đếm từ 1
    length = 1
    while length < idx:                  # tìm độ dài (lũy thừa 2) đủ chứa idx
        length *= 2
    shift = 0
    while idx > 1:
        half = length // 2
        if idx > half:                   # idx ở nửa sau
            idx -= half                  # quy về vị trí tương ứng ở nửa đầu
            shift += 1                   # nửa sau = nửa đầu + 1
        length = half
    return chr(ord("a") + shift % 26)


def offset_recursive(k: int) -> int:
    # độ lệch của ký tự thứ k (đếm từ 1) so với 'a'
    if k == 1:
        return 0
    half = 1 << ((k - 1).bit_length() - 1)     # lũy thừa 2 lớn nhất nhỏ hơn k
    return offset_recursive(k - half) + 1


def kth_recursive(k: int) -> str:
    return chr(ord("a") + offset_recursive(k) % 26)


for k, expected in examples:
    print(kth_halving(k), kth_recursive(k), "(kỳ vọng", expected, ")")

b b (kỳ vọng b )
c c (kỳ vọng c )


In [13]:
# Code nộp LeetCode cho Cách 3 (đổi tên lớp thành Solution khi nộp)
class SolutionHalving:
    def kthCharacter(self, k: int) -> str:
        length = 1
        while length < k:
            length *= 2
        shift = 0
        while k > 1:
            half = length // 2
            if k > half:
                k -= half
                shift += 1
            length = half
        return chr(ord("a") + shift % 26)


print(SolutionHalving().kthCharacter(5), SolutionHalving().kthCharacter(10))   # b c

b c


**Độ phức tạp Cách 3:** mỗi vòng giảm `length` đi một nửa nên có `O(log k)` vòng ⇒ **Time `O(log k)`**, **Space `O(1)`** (bản đệ quy dùng `O(log k)` ngăn xếp gọi hàm).

### Cách 4 — Công thức đếm bit `1`: `popcount(k - 1)`

**Giải thích.** Cách 3 thực chất mỗi lần "xoá bit cao nhất của `i = k - 1` và cộng 1". Số lần làm vậy đến khi `i = 0` chính là **số bit `1` trong `k - 1`**. Vậy:

> ký tự thứ `k` = `chr(ord("a") + popcount(k - 1) % 26)`

Hãy tự kiểm tra bảng `i = k - 1` (đếm từ 0), nhị phân và số bit `1`, so với `word` thật.

In [14]:
word = "abbcbccdbccdcdde"
print("i  nhị phân  popcount  chữ trong word  chữ theo công thức")
for i in range(16):
    pc = i.bit_count()
    print(f"{i:2d}  {i:04b}      {pc}         {word[i]}               {chr(97 + pc)}")
    assert word[i] == chr(97 + pc)

i  nhị phân  popcount  chữ trong word  chữ theo công thức
 0  0000      0         a               a
 1  0001      1         b               b
 2  0010      1         b               b
 3  0011      2         c               c
 4  0100      1         b               b
 5  0101      2         c               c
 6  0110      2         c               c
 7  0111      3         d               d
 8  1000      1         b               b
 9  1001      2         c               c
10  1010      2         c               c
11  1011      3         d               d
12  1100      2         c               c
13  1101      3         d               d
14  1110      3         d               d
15  1111      4         e               e


In [15]:
def kth_popcount(k: int) -> str:
    return chr(ord("a") + (k - 1).bit_count() % 26)


for k, expected in examples:
    print(kth_popcount(k), "(kỳ vọng", expected, ")")

b (kỳ vọng b )
c (kỳ vọng c )


In [16]:
# Code nộp LeetCode cho Cách 4 (đổi tên lớp thành Solution khi nộp)
class SolutionPopcount:
    def kthCharacter(self, k: int) -> str:
        return chr(ord("a") + (k - 1).bit_count() % 26)


print(SolutionPopcount().kthCharacter(5), SolutionPopcount().kthCharacter(10))   # b c

b c


**Độ phức tạp Cách 4:** đếm bit của một số `O(log k)` (hoặc coi như `O(1)` với số nguyên máy) ⇒ **Time `O(1)`/`O(log k)`**, **Space `O(1)`**.

Lưu ý: phần `% 26` không bao giờ cần với `k <= 500` (vì `popcount(k - 1) <= 9`), nhưng nó làm công thức đúng cho mọi `k` — ta sẽ thử ở mục 12 với `k = 2^26`.

## 8. TRACE từng bước

Ta trace `k = 10` theo Cách 3 (chia đôi chỉ số). Đây là bảng trạng thái:

| vòng | `idx` đầu vòng | `length` | `half` | `idx > half`? | `idx` sau | `shift` sau |
|---|---|---|---|---|---|---|
| 1 | 10 | 16 | 8 | có | 2 | 1 |
| 2 | 2 | 8 | 4 | không | 2 | 1 |
| 3 | 2 | 4 | 2 | không | 2 | 1 |
| 4 | 2 | 2 | 1 | có | 1 | 2 |

Hết vòng vì `idx = 1`; kết quả là `a` cộng `2` ⇒ `c`. (Vòng 2 và 3 "chỉ co" `length`, vì `idx` vẫn nằm ở nửa đầu.)

In [17]:
def trace_halving(k: int) -> str:
    idx = k
    length = 1
    while length < idx:
        length *= 2
    shift = 0
    print(f"k = {k}, length ban đầu = {length}")
    step = 0
    while idx > 1:
        step += 1
        half = length // 2
        in_second = idx > half
        print(f"  vòng {step}: idx={idx}, length={length}, half={half}, nửa sau? {in_second}", end="")
        if in_second:
            idx -= half
            shift += 1
        length = half
        print(f" -> idx={idx}, shift={shift}")
    ans = chr(ord("a") + shift % 26)
    print("  kết quả:", ans)
    return ans


assert trace_halving(10) == "c"
assert trace_halving(5) == "b"

k = 10, length ban đầu = 16
  vòng 1: idx=10, length=16, half=8, nửa sau? True -> idx=2, shift=1
  vòng 2: idx=2, length=8, half=4, nửa sau? False -> idx=2, shift=1
  vòng 3: idx=2, length=4, half=2, nửa sau? False -> idx=2, shift=1
  vòng 4: idx=2, length=2, half=1, nửa sau? True -> idx=1, shift=2
  kết quả: c
k = 5, length ban đầu = 8
  vòng 1: idx=5, length=8, half=4, nửa sau? True -> idx=1, shift=1
  kết quả: b


In [18]:
def trace_offsets(k: int) -> str:
    word = [0]
    print(f"k = {k}")
    ops = 0
    print("  ban đầu:", word)
    while len(word) < k:
        ops += 1
        word.extend([(x + 1) % 26 for x in word])
        print(f"  sau thao tác {ops}: dài {len(word):2d} ->", word)
    ans = chr(ord("a") + word[k - 1])
    print(f"  word[{k - 1}] = {word[k - 1]} -> {ans!r}")
    return ans


assert trace_offsets(10) == "c"

k = 10
  ban đầu: [0]
  sau thao tác 1: dài  2 -> [0, 1]
  sau thao tác 2: dài  4 -> [0, 1, 1, 2]
  sau thao tác 3: dài  8 -> [0, 1, 1, 2, 1, 2, 2, 3]
  sau thao tác 4: dài 16 -> [0, 1, 1, 2, 1, 2, 2, 3, 1, 2, 2, 3, 2, 3, 3, 4]
  word[9] = 2 -> 'c'


In [19]:
def trace_popcount(k: int) -> str:
    i = k - 1
    print(f"k = {k}, i = k - 1 = {i}, nhị phân = {i:b}")
    count = 0
    t = i
    while t:
        bit = t & 1                      # bit thấp nhất
        count += bit
        print(f"  t = {t:5b}, bit thấp = {bit}, count = {count}")
        t >>= 1                          # bỏ bit thấp nhất
    ans = chr(ord("a") + count % 26)
    print("  popcount =", count, "->", ans)
    return ans


assert trace_popcount(10) == "c"
assert trace_popcount(5) == "b"

k = 10, i = k - 1 = 9, nhị phân = 1001
  t =  1001, bit thấp = 1, count = 1
  t =   100, bit thấp = 0, count = 1
  t =    10, bit thấp = 0, count = 1
  t =     1, bit thấp = 1, count = 2
  popcount = 2 -> c
k = 5, i = k - 1 = 4, nhị phân = 100
  t =   100, bit thấp = 0, count = 0
  t =    10, bit thấp = 0, count = 0
  t =     1, bit thấp = 1, count = 1
  popcount = 1 -> b


## 9. Code nộp LeetCode

Ta chọn **Cách 4** (đếm bit) làm bản nộp vì ngắn nhất và đúng cho mọi `k`; nếu bạn dùng Python cũ chưa có `int.bit_count()` hãy thay bằng `bin(k - 1).count("1")`.

In [20]:
class Solution:
    def kthCharacter(self, k: int) -> str:
        # ký tự thứ k có độ lệch = số bit 1 của (k - 1), tính từ chữ 'a'
        return chr(ord("a") + (k - 1).bit_count() % 26)


sol = Solution()
print(sol.kthCharacter(5), sol.kthCharacter(10))     # b c

b c


## 10. Vì sao đúng

Gọi `f(i)` là độ lệch (so với `a`) của ký tự ở chỉ số `i` (đếm từ 0) trong `word` **vô hạn** (các `W_m` đều là tiền tố của nhau).

- **Cơ sở:** `f(0) = 0` (ký tự `a`) và `popcount(0) = 0`.
- **Bước quy nạp:** với `i >= 1`, đặt `2^m` là lũy thừa 2 lớn nhất `<= i`, tức `2^m <= i < 2^(m+1)`. Ký tự ở chỉ số `i` nằm trong nửa sau của `W_{m+1}`, nên `f(i) = f(i - 2^m) + 1`. Mà `i - 2^m` chính là `i` đã **bỏ bit cao nhất**, nên `popcount(i - 2^m) = popcount(i) - 1`.
- Theo giả thiết quy nạp `f(i - 2^m) = popcount(i - 2^m)`, suy ra `f(i) = popcount(i - 2^m) + 1 = popcount(i)`.

Cuối cùng ký tự thật là `a` cộng `f(i)` **theo modulo 26** (vì `z + 1 = a`). Với `k <= 500`, `f <= 9` nên không bao giờ quay vòng, nhưng công thức vẫn đúng cho `k` lớn nhờ `% 26`.

Với các cách khác: Cách 1, 2 mô phỏng đúng định nghĩa; Cách 3 là đúng lập luận quy nạp trên (mỗi vòng chính là một bước `f(i) = f(i - 2^m) + 1`).

## 11. Phân tích Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. Mô phỏng chuỗi | `O(k)` | `O(k)` |
| 2. Mô phỏng mảng độ lệch | `O(k)` | `O(k)` |
| 3. Chia đôi chỉ số | `O(log k)` | `O(1)` (đệ quy: `O(log k)`) |
| 4. `popcount(k - 1)` | `O(1)`/`O(log k)` | `O(1)` |

Với `k <= 500` mọi cách đều tức thì; nhưng chỉ có Cách 3 và 4 còn dùng được khi `k` lên tới `10^18` (không thể dựng `word` dài `10^18`).

## 12. Bộ test

Ta gom mọi cách vào `impls`, so với kỳ vọng.

In [21]:
import random

impls = {
    "simulate_string": kth_simulate_string,
    "offsets": kth_offsets,
    "halving": kth_halving,
    "recursive": kth_recursive,
    "popcount": kth_popcount,
    "leetcode": Solution().kthCharacter,
}


def check_all(k, expected, label):
    for name, fn in impls.items():
        actual = fn(k)
        assert actual == expected, (label, name, k, actual, expected)
    print("PASS", label)


for idx, (k, expected) in enumerate(examples, 1):
    check_all(k, expected, f"ví dụ {idx}")

PASS ví dụ 1
PASS ví dụ 2


### 12.1. Các trường hợp biên

Giá trị nhỏ nhất `k = 1`, các mốc lũy thừa của 2 (`k = 2, 4, 8, ...`), mốc ngay sau chúng (`k = 3, 5, 9, ...`), và giá trị lớn nhất `k = 500`. Kỳ vọng được tính từ chuỗi dựng sẵn ở mục 4 (độc lập với thuật toán).

In [22]:
word16 = "abbcbccdbccdcdde"
check_all(1, "a", "k = 1")
check_all(2, "b", "k = 2")
check_all(3, "b", "k = 3")
check_all(4, "c", "k = 4")
check_all(8, "d", "k = 8, cuối word 3 thao tác")
check_all(16, "e", "k = 16, cuối word 4 thao tác")
check_all(13, "c", "k = 13 (bài tập 2)")
for k in range(1, 17):
    check_all(k, word16[k - 1], f"k = {k} khớp word dựng tay")

PASS k = 1
PASS k = 2
PASS k = 3
PASS k = 4
PASS k = 8, cuối word 3 thao tác
PASS k = 16, cuối word 4 thao tác
PASS k = 13 (bài tập 2)
PASS k = 1 khớp word dựng tay
PASS k = 2 khớp word dựng tay
PASS k = 3 khớp word dựng tay
PASS k = 4 khớp word dựng tay
PASS k = 5 khớp word dựng tay
PASS k = 6 khớp word dựng tay
PASS k = 7 khớp word dựng tay
PASS k = 8 khớp word dựng tay
PASS k = 9 khớp word dựng tay
PASS k = 10 khớp word dựng tay
PASS k = 11 khớp word dựng tay
PASS k = 12 khớp word dựng tay
PASS k = 13 khớp word dựng tay
PASS k = 14 khớp word dựng tay
PASS k = 15 khớp word dựng tay
PASS k = 16 khớp word dựng tay


In [23]:
# k lớn nhất theo ràng buộc: so với dựng word đầy đủ độc lập
word = [0]
while len(word) < 500:
    word += [(x + 1) % 26 for x in word]
check_all(500, chr(97 + word[499]), "k = 500")
check_all(256, chr(97 + word[255]), "k = 256")
check_all(257, chr(97 + word[256]), "k = 257")
print("popcount(499) =", (499).bit_count(), "->", chr(97 + (499).bit_count()))

PASS k = 500
PASS k = 256
PASS k = 257
popcount(499) = 7 -> h


### 12.2. Phản ví dụ cho các cách làm sai

Ba lỗi hay gặp: (a) dùng `word[k]` thay vì `word[k - 1]`; (b) dùng `popcount(k)` thay vì `popcount(k - 1)`; (c) cộng 1 vào ký tự mà không quay vòng `z`. Ta chỉ ra ví dụ cụ thể làm chúng sai.

In [24]:
def wrong_index(k):
    word = "a"
    while len(word) <= k:                # cần đủ dài để word[k] hợp lệ
        word += shift_string(word)
    return word[k]                       # SAI: ký tự thứ k nằm ở chỉ số k - 1


def wrong_popcount(k):
    return chr(ord("a") + k.bit_count())  # SAI: phải là (k - 1).bit_count()


def wrong_no_wrap(ch):
    return chr(ord(ch) + 1)              # SAI với 'z'


print(wrong_index(5), "!=", Solution().kthCharacter(5))          # c != b
print(wrong_popcount(5), "!=", Solution().kthCharacter(5))       # c != b
print(repr(wrong_no_wrap("z")), "!=", repr(shift_string("z")))   # '{' != 'a'
assert wrong_index(5) != Solution().kthCharacter(5)
assert wrong_popcount(5) != Solution().kthCharacter(5)
assert wrong_no_wrap("z") != shift_string("z")

c != b
c != b
'{' != 'a'


### 12.3. Đối chiếu nhiều `k` và `k` vượt ràng buộc

(1) Với **mọi** `k` từ 1 đến 500 so tất cả các cách (kỳ vọng từ chuỗi dựng độc lập). (2) Một số `k` ngẫu nhiên có seed, trong ràng buộc. (3) Vì ta **không thể dựng** `word` rất dài, với `k` lớn ngoài ràng buộc (chỉ để xem công thức quay vòng `% 26` làm việc) ta so Cách 3 với Cách 4, và mô phỏng mảng độ lệch cho `k` tới `2^15`.

In [25]:
# (1) toàn bộ k = 1..500
ref = [0]
while len(ref) < 500:
    ref += [(x + 1) % 26 for x in ref]
for k in range(1, 501):
    expected = chr(97 + ref[k - 1])
    for name, fn in impls.items():
        assert fn(k) == expected, (name, k)
print("PASS 500 giá trị k")

# (2) ngẫu nhiên, có seed
rng = random.Random(3304)
for _ in range(300):
    k = rng.randint(1, 500)
    expected = chr(97 + ref[k - 1])
    assert all(fn(k) == expected for fn in impls.values()), k
print("PASS 300 k ngẫu nhiên")

# (3) vượt ràng buộc: k tới 2^15 bằng mô phỏng độ lệch; quay vòng với k = 2^26
big = [0]
while len(big) < 2 ** 15:
    big += [(x + 1) % 26 for x in big]
for k in range(1, 2 ** 15 + 1, 37):
    assert kth_halving(k) == kth_popcount(k) == kth_recursive(k) == chr(97 + big[k - 1]), k
k = 2 ** 26                              # popcount(k - 1) = 26 -> quay một vòng đầy đủ về 'a'
print(kth_halving(k), kth_popcount(k), (k - 1).bit_count())   # a a 26
assert kth_halving(k) == kth_popcount(k) == "a"
assert kth_halving(2 ** 26 - 1) == kth_popcount(2 ** 26 - 1) == "z"   # popcount = 25
print("PASS k vượt ràng buộc")

PASS 500 giá trị k
PASS 300 k ngẫu nhiên
a a 26
PASS k vượt ràng buộc


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Lệch chỉ số `k` và `k - 1`

In [26]:
def sai_lech_chi_so(k):
    word = "a"
    while len(word) < k:
        word += shift_string(word)
    return word[k]                       # SAI: có thể IndexError, hoặc lấy nhầm ký tự


try:
    print("k=5:", sai_lech_chi_so(5), "| đúng:", Solution().kthCharacter(5))
    print("k=8:", sai_lech_chi_so(8), "| đúng:", Solution().kthCharacter(8))
except IndexError as e:
    print("IndexError:", e)

k=5: c | đúng: b
IndexError: string index out of range


**Vì sao sai:** đề đếm ký tự từ **1** nhưng Python đánh chỉ số từ **0**. Với `k = 5` ký tự thứ 5 nằm ở `word[4]`. Dùng `word[k]` lấy nhầm ký tự kế (`c` thay vì `b`), và với `k = 8` (đúng bằng độ dài) còn **ra ngoài mảng** (`IndexError`). **Cách sửa:** luôn dùng `word[k - 1]`.

### Lỗi 2: Quên quay vòng `z → a`

In [27]:
def sai_khong_quay_vong(w):
    return "".join(chr(ord(ch) + 1) for ch in w)    # SAI: 'z' + 1 thành '{'


def dung_quay_vong(w):
    return "".join(chr((ord(ch) - 97 + 1) % 26 + 97) for ch in w)


print(repr(sai_khong_quay_vong("zb")), "| đúng:", repr(dung_quay_vong("zb")))

'{c' | đúng: 'ac'


**Vì sao sai:** mã của `z` là 122, cộng 1 thành 123 là ký tự `{`, không phải chữ cái. Với `k <= 500` bạn chưa chạm tới `z` (chữ lớn nhất là `j`), nhưng một lời giải đúng nên xử lý việc này; ở các bản mở rộng (Game II) nó xuất hiện thật. **Cách sửa:** làm việc với độ lệch `0..25` và dùng `% 26`.

### Lỗi 3: Vừa lặp vừa nối vào chính list đang lặp

In [28]:
def sai_vua_lap_vua_noi(k):
    word = [0]
    while len(word) < k:
        for x in word:                   # SAI: word dài ra trong khi đang duyệt -> không bao giờ kết thúc
            word.append((x + 1) % 26)
            if len(word) > 40:           # chốt chặn an toàn để demo không treo máy
                return "vòng lặp chạy vô hạn (đã bị chặn), độ dài = " + str(len(word))
    return word[k - 1]


print(sai_vua_lap_vua_noi(5))
print("đúng:", Solution().kthCharacter(5))

vòng lặp chạy vô hạn (đã bị chặn), độ dài = 41
đúng: b


**Vì sao sai:** `for x in word` duyệt theo danh sách **đang lớn dần**; mỗi lần `append` thêm một phần tử nữa để duyệt, nên vòng `for` không bao giờ dừng (và độ dài vượt quá ý muốn). **Cách sửa:** tạo list mới bằng comprehension rồi mới `extend` (như Cách 2), hoặc duyệt trên **bản sao** `for x in word[:]`.

### Lỗi 4: Công thức bit mà quên trừ 1

In [29]:
def sai_quen_tru_1(k):
    return chr(ord("a") + k.bit_count())  # SAI: phải là (k - 1).bit_count()


bad = [k for k in range(1, 20) if sai_quen_tru_1(k) != Solution().kthCharacter(k)]
print("k bị sai trong 1..19:", bad)
print("k=1:", sai_quen_tru_1(1), "| đúng:", Solution().kthCharacter(1))

k bị sai trong 1..19: [1, 3, 4, 5, 7, 8, 9, 11, 12, 13, 15, 16, 17, 19]
k=1: b | đúng: a


**Vì sao sai:** ký tự đầu tiên (`k = 1`) có chỉ số `i = 0` và độ lệch `0` (`a`); `popcount(1) = 1` cho `b` — sai ngay từ chữ đầu. Công thức đúng làm việc với chỉ số đếm từ **0**, tức `k - 1`. **Cách sửa:** viết `(k - 1).bit_count()` và thử ngay `k = 1`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Xâu được **nhân đôi**: `W' = W + f(W)`" và hỏi vị trí **rất xa** ⇒ đừng dựng xâu; hãy **lần ngược** bằng cách chia đôi chỉ số (đệ quy / vòng lặp). Đây là họ bài *K-th Symbol in Grammar*, *Thue–Morse*.
- Mỗi lần "rơi vào nửa sau" ứng với **một bit 1** của chỉ số (đếm từ 0). Bài nào có cấu trúc nhân đôi thường có lời giải theo bit.
- Luôn tự hỏi: đề đếm từ 0 hay từ 1? Python đếm từ 0, đề đếm từ 1 ⇒ cần `k - 1`.
- Dùng độ lệch `0..25` và `% 26` thay cho mã ký tự khi cần cộng/trừ chữ cái — tránh lỗi `z`.
- Khi ràng buộc nhỏ (`k <= 500`), cách mô phỏng thẳng tay là đủ; nhưng hãy biết cách lên cấp để không bị bất ngờ ở bài Medium cùng họ.

## 15. Biến thể của bài

1. **Game II (LeetCode 3307, Medium):** cùng ý tưởng nhưng `k` rất lớn (cỡ `10^14`) và mỗi bước có thể là "nối chuỗi y nguyên" hoặc "nối chuỗi đã dịch" tuỳ mảng thao tác — cần lần ngược theo bit. (Chi tiết đề: đối chiếu lại đề gốc.)
2. **K-th Symbol in Grammar (779):** nhân đôi với luật `0 → 01`, `1 → 10`; đáp án là **chẵn/lẻ** của số bit 1 — chính là dãy **Thue–Morse**.
3. **Mức dịch `d` bất kỳ:** mỗi lần thao tác dịch `d` chữ thay vì 1 ⇒ độ lệch `d * popcount(k - 1)`.
4. **Đếm chữ:** sau `m` thao tác có bao nhiêu ký tự bằng chữ thứ `j`? Là hệ số nhị thức `C(m, j)` (khi `m < 26`).

In [30]:
import math

# (2) dãy Thue-Morse: số bit 1 chẵn -> 0, lẻ -> 1
print([i.bit_count() % 2 for i in range(16)])       # [0,1,1,0,1,0,0,1,1,0,0,1,0,1,1,0]


# (3) dịch d chữ mỗi lần
def kth_shift_d(k, d):
    return chr(ord("a") + d * (k - 1).bit_count() % 26)


def brute_shift_d(k, d):
    word = [0]
    while len(word) < k:
        word += [(x + d) % 26 for x in word]
    return chr(97 + word[k - 1])


assert all(kth_shift_d(k, d) == brute_shift_d(k, d) for k in range(1, 300) for d in (1, 2, 5, 25))
print(kth_shift_d(5, 2), kth_shift_d(10, 13))        # c  a   (d=2: 2*1=2 ; d=13: 13*2=26 -> 0)


# (4) đếm chữ thứ j sau m thao tác bằng C(m, j)
def count_letter(m, j):
    return math.comb(m, j)


word = [0]
for _ in range(6):
    word += [(x + 1) % 26 for x in word]
for j in range(7):
    assert word.count(j) == count_letter(6, j)
print([word.count(j) for j in range(7)])             # [1, 6, 15, 20, 15, 6, 1]

[0, 1, 1, 0, 1, 0, 0, 1, 1, 0, 0, 1, 0, 1, 1, 0]
c a
[1, 6, 15, 20, 15, 6, 1]


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Mô phỏng chuỗi | làm đúng từng chữ đề | `O(k)` | `O(k)` | để hiểu đề, đối chiếu |
| 2. Mảng độ lệch | cùng ý, dùng số 0..25 | `O(k)` | `O(k)` | code ngắn, an toàn với `k <= 500` |
| 3. Chia đôi chỉ số | nửa sau = nửa đầu + 1, lần ngược | `O(log k)` | `O(1)` | khi `k` rất lớn, dạng bài Game II |
| 4. `popcount(k - 1)` | mỗi lần vào nửa sau = 1 bit 1 | `O(1)`/`O(log k)` | `O(1)` | **nên nộp**, ngắn nhất |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: `ord("c") - ord("a")`, `chr(97 + 27 % 26)`, `(7).bit_count()`, `Solution().kthCharacter(1)`, `Solution().kthCharacter(6)`.

**Bài 2 (tính tay).** Viết `word` sau 4 thao tác (16 ký tự), rồi đọc ký tự thứ 13. Sau đó kiểm tra bằng công thức `popcount(12)`.

**Bài 3 (điền TODO).** Hoàn thành `kth_popcount_todo(k)`.

In [31]:
def kth_popcount_todo(k):
    # TODO 1: tính i = k - 1
    # TODO 2: đếm số bit 1 của i (dùng i.bit_count())
    # TODO 3: trả về chữ cái 'a' cộng số đó (nhớ % 26)
    pass

# KHÔNG gọi hàm TODO ở đây để khi Run All không bị lỗi (TODO chưa điền nên kết quả chưa đúng).

**Bài 4 (tự viết lại).** Viết `kth_by_recursion(k)` **đệ quy thuần**: nếu `k == 1` trả `"a"`. Ngược lại gọi `half` là lũy thừa 2 lớn nhất **nhỏ hơn** `k`; ký tự thứ `k` là ký tự thứ `k - half` rồi cộng 1 (nhớ quay vòng `z → a`). Gợi ý: viết một hàm trợ giúp `next_char(ch)`.

**Bài 5 (biến thể / mini challenge).** (a) Cho luật mới: mỗi thao tác **dịch 2 chữ** thay vì 1. Viết `kth_char_step2(k)`. (b) Viết `count_char(m, ch)` trả về số lần chữ `ch` xuất hiện trong `word` sau `m` thao tác (`m <= 10`, nên không cần quay vòng).

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `2`, `"b"`, `3`, `"a"`, `"c"` (`k=6` → chỉ số 5 = `101` có 2 bit 1 → `c`). Bài 3: `i = k - 1`, rồi `i.bit_count()`.
- **Mức 2 (vừa):** Bài 2: nửa đầu `abbcbccd`, nửa sau `bccdcdde`; ký tự thứ 13 là chữ thứ 5 của nửa sau cộng 1 từ chữ thứ 5 của nửa đầu `b`... hãy tự đếm cẩn thận. Bài 4: kết quả đệ quy là `next_char(kth_by_recursion(k - half))`.
- **Mức 3 (sâu):** Bài 5(a): độ lệch là `2 * popcount(k - 1)`. Bài 5(b): số lần xuất hiện của độ lệch `j` sau `m` thao tác là `C(m, j)`, với `j = ord(ch) - ord("a")`.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [32]:
# Bài 1
assert ord("c") - ord("a") == 2
assert chr(97 + 27 % 26) == "b"
assert (7).bit_count() == 3
assert Solution().kthCharacter(1) == "a"
assert Solution().kthCharacter(6) == "c"

# Bài 2
w16 = "abbcbccdbccdcdde"
assert len(w16) == 16 and w16[13 - 1] == "c" == chr(97 + (12).bit_count())


# Bài 3
def kth_popcount_ans(k):
    i = k - 1
    return chr(ord("a") + i.bit_count() % 26)


assert all(kth_popcount_ans(k) == Solution().kthCharacter(k) for k in range(1, 501))


# Bài 4
def next_char(ch):
    return chr((ord(ch) - 97 + 1) % 26 + 97)


def kth_by_recursion(k):
    if k == 1:
        return "a"
    half = 1 << ((k - 1).bit_length() - 1)
    return next_char(kth_by_recursion(k - half))


assert all(kth_by_recursion(k) == Solution().kthCharacter(k) for k in range(1, 501))
assert next_char("z") == "a"


# Bài 5(a)
def kth_char_step2(k):
    return chr(ord("a") + 2 * (k - 1).bit_count() % 26)


def brute_step2(k):
    w = [0]
    while len(w) < k:
        w += [(x + 2) % 26 for x in w]
    return chr(97 + w[k - 1])


assert all(kth_char_step2(k) == brute_step2(k) for k in range(1, 501))


# Bài 5(b)
def count_char(m, ch):
    return math.comb(m, ord(ch) - ord("a"))


for m in range(0, 11):
    w = "a"
    for _ in range(m):
        w += shift_string(w)
    for ch in "abcdefghijk":
        assert w.count(ch) == count_char(m, ch), (m, ch)
print("PASS toàn bộ đáp án")

PASS toàn bộ đáp án


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (đếm bit của `k - 1`)

```python
class Solution:
    def kthCharacter(self, k: int) -> str:
        return chr(ord("a") + (k - 1).bit_count() % 26)
```

**Template dự phòng** (mô phỏng, dễ nhớ nhất)

```python
class Solution:
    def kthCharacter(self, k: int) -> str:
        word = [0]
        while len(word) < k:
            word.extend([(x + 1) % 26 for x in word])
        return chr(ord("a") + word[k - 1])
```

**Checklist trước khi submit**

- [ ] Tôi đã dùng `k - 1` (chỉ số đếm từ 0) khi đọc ký tự / tính bit?
- [ ] Tôi có quay vòng `z → a` bằng `% 26` không?
- [ ] Tôi không vừa lặp vừa nối vào list đang lặp?
- [ ] Hàm trả về **chuỗi một ký tự** (`str`), không phải số?
- [ ] Đã chạy ví dụ `k = 5 -> "b"`, `k = 10 -> "c"`, và biên `k = 1`, `k = 500`?
- [ ] Python của LeetCode có `int.bit_count()` (3.10+); nếu lo thì dùng `bin(k - 1).count("1")`.